# Анализ оттока клиентов (Churn Prediction)

**BI-проект:** Исследование причин оттока, когортный анализ, LTV, предсказательная модель и рекомендации по удержанию.

Цель — помочь компании снизить отток клиентов и повысить удержание.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

import warnings
warnings.filterwarnings('ignore')

print("Библиотеки загружены!")

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print("Размер датасета:", df.shape)
df.head()

In [ ]:
# Преобразование типов
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Churn в 0/1
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Данные после предобработки:")
print(df.isnull().sum().sum())  # должно быть 0

## 1. Exploratory Data Analysis (EDA)

In [ ]:
print("Общая доля оттока:", round(df['Churn'].mean() * 100, 2), "%\n")

print("Отток по полу:")
print(df.groupby('gender')['Churn'].mean().round(3))

print("\nОтток по SeniorCitizen:")
print(df.groupby('SeniorCitizen')['Churn'].mean().round(3))

print("\nОтток по наличию партнёра:")
print(df.groupby('Partner')['Churn'].mean().round(3))

print("\nОтток по типу контракта:")
print(df.groupby('Contract')['Churn'].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(x='Contract', y='Churn', data=df, ax=axes[0])
axes[0].set_title('Отток по типу контракта')

sns.barplot(x='InternetService', y='Churn', data=df, ax=axes[1])
axes[1].set_title('Отток по типу интернета')

sns.barplot(x='SeniorCitizen', y='Churn', data=df, ax=axes[2])
axes[2].set_title('Отток среди пенсионеров')

plt.tight_layout()
plt.show()

## 2. Когортный анализ и Retention

In [ ]:
# Создаём когорту по месяцу начала (tenure)
df['Cohort'] = (df['tenure'] // 6) * 6   # группируем по 6 месяцев

cohort = df.groupby(['Cohort', 'tenure'])['customerID'].count().unstack(0)

# Retention rate
retention = cohort.div(cohort.iloc[0], axis=1)

print("Retention по когортам (первые 12 месяцев):")
print(retention.head(13))

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(retention.head(12), annot=True, fmt='.2%', cmap='YlGnBu')
plt.title('Retention Rate по когортам')
plt.ylabel('Tenure (месяцы)')
plt.xlabel('Когорта (месяцы с начала)')
plt.show()

## 3. Расчёт LTV (Lifetime Value)

In [ ]:
# Простой LTV = MonthlyCharges * tenure (для ушедших) + прогноз для активных
df['LTV'] = df['MonthlyCharges'] * df['tenure']

print("Средний LTV:")
print("Общий:", round(df['LTV'].mean(), 2))
print("У оттока:", round(df[df['Churn'] == 1]['LTV'].mean(), 2))
print("У активных:", round(df[df['Churn'] == 0]['LTV'].mean(), 2))

## 4. A/B-тестирование гипотез

In [ ]:
# Сравниваем отток по разным группам
ab_test = df.groupby(['Contract', 'InternetService']).agg({
    'Churn': ['mean', 'count']
}).round(3)

print("A/B-тест: Отток по типу контракта и интернету")
print(ab_test)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Contract', y='Churn', hue='InternetService', data=df)
plt.title('A/B-тест: Отток по контракту и типу интернета')
plt.ylabel('Доля оттока')
plt.show()

## 5. Бизнес-рекомендации

1. **Тип контракта** — клиенты на Month-to-month уходят значительно чаще. Нужно стимулировать переход на годовые/двухлетние контракты (скидки, бонусы).

2. **Интернет-сервис** — Fiber optic имеет самый высокий отток. Проверить качество услуги.

3. **SeniorCitizen** — пенсионеры уходят чаще — нужна специальная программа лояльности.

4. **LTV** — активные клиенты имеют значительно выше LTV. Фокус на удержании.

5. **Действия**:
   - Внедрить программу удержания для клиентов с высоким риском оттока
   - Предлагать скидки на годовые контракты
   - Улучшить качество Fiber optic

## 6. Построение модели предсказания оттока

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# Выбираем признаки
features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 
            'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

# Преобразуем категориальные в dummy
X = pd.get_dummies(df[features], drop_first=True)
y = df['Churn']

# Разделяем на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Данные подготовлены для модели.")

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Предсказание
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Матрица ошибок')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Итоговые выводы и рекомендации

Проект показал:
- Высокий отток на ежемесячных контрактах
- Fiber optic — зона риска
- Модель предсказания оттока готова к использованию

**Рекомендации:**
- Стимулировать долгосрочные контракты
- Улучшить качество Fiber optic
- Внедрить систему раннего предупреждения оттока